# Threshold Analysis
Sweep threshold values and plot ROC / Precision-Recall curves to justify the 99th-percentile choice.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml
from sklearn.metrics import roc_curve, precision_recall_curve, auc

from src.models.lstm_autoencoder import GRUAutoencoder
from src.data.preprocessor import load_scaler
from src.data.ciciot_loader import load_dataset, make_windows

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
dcfg = cfg['data']['ciciot']
mcfg = cfg['model']

In [ ]:
_, _, X_test_raw, y_test = load_dataset(
    csv_path='../'+dcfg['raw_path'],
    scaler_path='../'+dcfg['scaler_path'],
)
scaler = load_scaler('../'+dcfg['scaler_path'])
X_test_windows = make_windows(scaler.transform(X_test_raw), mcfg['window_size'])
y_windows = y_test[mcfg['window_size']-1 : mcfg['window_size']-1+len(X_test_windows)]

model = GRUAutoencoder(
    input_size=X_test_windows.shape[2],
    hidden_size=mcfg['hidden_size'],
    bottleneck_size=mcfg['bottleneck_size'],
    seq_len=mcfg['window_size'],
)
model.load_state_dict(torch.load('../'+cfg['saved_models']['ciciot_model'], map_location='cpu'))
model.eval()

with torch.no_grad():
    errors = model.reconstruction_error(torch.tensor(X_test_windows, dtype=torch.float32)).numpy()

In [ ]:
fpr, tpr, _ = roc_curve(y_windows, errors)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curve — CIC IoT')
plt.legend()
plt.show()

In [ ]:
prec, rec, thresholds = precision_recall_curve(y_windows, errors)
f1 = 2 * prec * rec / (prec + rec + 1e-8)
best_idx = f1.argmax()
print(f'Best F1 threshold: {thresholds[best_idx]:.6f}, F1={f1[best_idx]:.4f}')

for pct in [95, 97, 99, 99.5]:
    t = np.percentile(errors[y_windows == 0], pct)
    pred = (errors > t).astype(int)
    acc = (pred == y_windows).mean()
    print(f'Percentile {pct:5.1f}%: threshold={t:.6f}, accuracy={acc:.4f}')